In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A5-2
- note : test pin shoud be connected to positive terminal on DMM
- kt : VDD3V3
- ps.ch1 : relay
- ks : ---> relay.ch1 (VIN)
- ps.ch2 : ---> dm.ch3 (I_INTB / I_OC_FAULT)
- dm.ch3 : ---> INTB / OC_FAULT pin
- dm.ch4 : ---> dm.ch3 (V_INTB / V_OC_FAULT) and GND
- relay.ch1 : VIN

In [ ]:
temperature = "25C"

log.output_set_filename(f"[{temperature}_5-2] VOL, ILK - OC_FAULT")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "V_OC_FAULT (V)", "I_OC_FAULT (uA)"])

In [ ]:
disable_all_ps()
relay.init_channels



v_start  = 0.1
v_finish = 3.31

list_vin = [5, 20, 28]
v_vdd = 3.3

list_temp = list(np.arange(v_start, v_finish, 0.1))
list_ocf  = [round(num, 1) for num in list_temp]

print(list_ocf)

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

ps.ch2.cfg_all = list_ocf[0], 0.1
ps.ch2.enable
delay(0.5)

ks.cfg_all = list_vin[0], 0.5 # vin
ks.enable
delay(0.5)

relay.ch1.enable # vin
delay(0.5)

In [ ]:
# ----------------------------------------------------------------------------

for vin in list_vin:
    
    ks.vset = vin
    
    for v_intb in list_ocf:
        
        ps.ch2.vset = v_intb
        
        v_ocf = round(dm.ch4.voltage_20V, 6)
        i_ocf = round(dm.ch3.current_2mA * 1e+6, 6)
        
        ret = [vin, v_vdd, v_ocf, i_ocf]
        log.output_csv(ret)
        
        print(ret)
    
# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()